# Lab 1: Simple Object Detection and Benchmarking with OpenVINO

**Goals:**
- Run object detection using OpenVINO
- Understand inference flow
- Measure throughput and FPS

## Installation

Run the cell below once to install required libraries.

In [ ]:
# Install required libraries (run once)
!pip install openvino opencv-python numpy matplotlib ipywidgets

## Simple Pipeline

**Input Image** → **Preprocessing** → **OpenVINO Model** → **Detection Output** → **Bounding Boxes**

## Setup

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt
import openvino as ov
import openvino.properties.hint as hints
import time
import os
import glob
from ipywidgets import Dropdown
from IPython.display import display

# Configuration
IMAGE_PATH = "media/sample_image-1.jpg"
MODEL_NAME = "ATSS-MobileNetV2"
PRECISION = "FP16"

device_dropdown = Dropdown(options=["CPU", "GPU", "NPU"], value="CPU", description="Device:")
display(device_dropdown)

In [ ]:
# Create dummy image if not present (for local testing)
os.makedirs("media", exist_ok=True)
if not os.path.exists(IMAGE_PATH):
    # Synthetic road-like image (640x480) similar to sample_image-1.jpg
    h, w = 480, 640
    img = np.zeros((h, w, 3), dtype=np.uint8)
    img[:] = (90, 90, 95)  # Gray road
    cv2.rectangle(img, (100, 200), (250, 350), (40, 40, 40), -1)   # Dark car
    cv2.rectangle(img, (350, 220), (500, 360), (180, 180, 180), -1)  # Light car
    cv2.rectangle(img, (200, 280), (320, 400), (0, 0, 150), -1)     # Red car
    cv2.imwrite(IMAGE_PATH, img)
    print(f"Created dummy image: {IMAGE_PATH}")

## Model Loading

In [ ]:
def find_model_xml(path):
    """Find .xml file in the given path."""
    files = glob.glob(os.path.join(path, "*.xml"))
    if not files:
        raise FileNotFoundError(f"No .xml model found in {path}")
    return files[0]

model_dir = os.path.join("models", MODEL_NAME, PRECISION)
xml_path = find_model_xml(model_dir)

core = ov.Core()
model = core.read_model(xml_path)
compiled_model = core.compile_model(model, device_dropdown.value)

input_layer = compiled_model.input(0)
input_shape = input_layer.shape
print(f"Model loaded: {xml_path}")
print(f"Device: {device_dropdown.value}")
print(f"Input shape: {input_shape}")

## Image Inference

In [ ]:
def preprocess_image(img, input_shape):
    """Resize and convert to NCHW format."""
    _, _, h, w = input_shape
    resized = cv2.resize(img, (w, h))
    nchw = np.expand_dims(resized.transpose(2, 0, 1), axis=0).astype(np.float32)
    return nchw

def run_inference(compiled_model, input_tensor):
    """Run synchronous inference."""
    return compiled_model(input_tensor)

def postprocess(output, orig_h, orig_w, thresh=0.5):
    """Parse detection output to boxes [x1, y1, x2, y2]."""
    result = list(output.values())[0]
    result = np.squeeze(result)
    if result.ndim == 1:
        result = np.expand_dims(result, 0)
    # Assume format [N, 5] or [N, 6]: x1, y1, x2, y2, conf, [label]
    boxes = []
    for det in result:
        if len(det) >= 5 and det[4] > thresh:
            x1, y1, x2, y2 = det[0], det[1], det[2], det[3]
            # Scale to original size if normalized
            if x2 <= 1 and y2 <= 1:
                x1, y1, x2, y2 = x1*orig_w, y1*orig_h, x2*orig_w, y2*orig_h
            boxes.append([int(x1), int(y1), int(x2), int(y2)])
    return np.array(boxes) if boxes else np.zeros((0, 4))

def draw_boxes(img, boxes, color=(0, 200, 0), thickness=2):
    """Draw bounding boxes on image."""
    out = img.copy()
    for box in boxes:
        x1, y1, x2, y2 = box
        cv2.rectangle(out, (x1, y1), (x2, y2), color, thickness)
    return out

In [ ]:
image = cv2.imread(IMAGE_PATH)
if image is None:
    raise FileNotFoundError(f"Image not found: {IMAGE_PATH}")

orig_h, orig_w = image.shape[:2]
input_tensor = preprocess_image(image, input_shape)
output = run_inference(compiled_model, input_tensor)
boxes = postprocess(output, orig_h, orig_w)
result_img = draw_boxes(image, boxes)

plt.figure(figsize=(10, 6))
plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title(f"Detections: {len(boxes)} objects")
plt.show()

## Optional: PyTorch to OpenVINO Conversion

For reference only. The main workflow uses pre-converted IR models. Use `convert_model()` and `ov.save_model()` to export PyTorch models to OpenVINO IR.

## Benchmarking Throughput

Run inference with 2 and 4 streams to measure FPS and throughput. This demonstrates scalability rather than deep optimization.

In [ ]:
def benchmark(compiled_model, input_tensor, num_iter=100):
    """Run inference loop; return FPS (images/sec)."""
    for _ in range(10):
        compiled_model(input_tensor)
    start = time.perf_counter()
    for _ in range(num_iter):
        compiled_model(input_tensor)
    return num_iter / (time.perf_counter() - start)

results = {}
for num_streams in [2, 4]:
    config = {
        hints.performance_mode: hints.PerformanceMode.THROUGHPUT,
        hints.num_requests: str(num_streams)
    }
    compiled = core.compile_model(model, device_dropdown.value, config)
    fps = benchmark(compiled, input_tensor)
    results[num_streams] = fps
    print(f"Streams: {num_streams} → FPS: {fps:.1f}")

# Charts
streams = list(results.keys())
vals = [results[s] for s in streams]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.bar(streams, vals, color="steelblue", edgecolor="black")
ax1.set_xlabel("Number of Streams")
ax1.set_ylabel("FPS (images/sec)")
ax1.set_title("FPS by Streams")
ax1.set_xticks(streams)

ax2.bar(streams, vals, color="seagreen", edgecolor="black")
ax2.set_xlabel("Number of Streams")
ax2.set_ylabel("Throughput (images/sec)")
ax2.set_title("Throughput by Streams")
ax2.set_xticks(streams)

plt.tight_layout()
plt.show()

## Wrap-up

- OpenVINO simplifies deployment of object detection models
- The detection pipeline is straightforward: load → preprocess → infer → postprocess
- Throughput scales with streams (parallel inference requests)
- Pre-converted IR models keep the workflow simple and fast